# 03 - Evaluation: where does the model fail, and can it tell?

This notebook answers steps 3 and 4 of `docs/plan.md` on the validation images:

1. Turn the pixel classes of the U-Net into single nuclei (watershed from the interior class).
2. Score every image with the Data Science Bowl metric (average precision over IoU 0.5 to 0.95) and F1 at IoU 0.5.
3. Report the scores per image type (H1) and count merged and split nuclei (H2).
4. Give every image one uncertainty number (mean pixel entropy) and test whether the most uncertain 10 % of the images contain the worst 10 % (H3).

Kaggle setup: Accelerator GPU T4, Internet on, and **+ Add Input > Your Work > the saved version of notebook 02**, so that `unet_best.pt` is available under `/kaggle/input`.
Output: `out/per_image.csv`, `out/by_type.csv`, `out/warning.json`, three figures, zipped as `results_03.zip`.

In [ ]:
import glob, zipfile, urllib.request, io, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io as skio
from skimage.segmentation import find_boundaries
import torch
import torch.nn as nn
import torch.nn.functional as F

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
DATA = Path('/tmp/nuclei_data')   # raw data is not saved as output
OUT = WORK / 'out'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

URL = 'https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip'
zip_path, train_dir = DATA / 'stage1_train.zip', DATA / 'stage1_train'
if not zip_path.exists():
    urllib.request.urlretrieve(URL, zip_path)
if not train_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(train_dir)

SPLIT_URL = 'https://raw.githubusercontent.com/veliatees/nuclei-segmentation/main/results/split.csv'
split = pd.read_csv(io.BytesIO(urllib.request.urlopen(SPLIT_URL).read()))
paths = {Path(p).stem: p for p in glob.glob(str(train_dir / '**' / 'images' / '*.png'), recursive=True)}
val = split[(split.split == 'val') & split.id.isin(paths)].reset_index(drop=True)
print(val.type.value_counts())

Same loading and normalisation as in notebook 02. Here the label image keeps every nucleus as its own number, because the evaluation compares single nuclei.

In [ ]:
def load(img_path):
    img = skio.imread(img_path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    label = np.zeros(img.shape[:2], np.int32)
    for k, mp in enumerate(sorted(glob.glob(str(Path(img_path).parent.parent / 'masks' / '*.png'))), start=1):
        label[skio.imread(mp) > 0] = k
    return img, label

def normalise(img):
    x = img.astype(np.float32) / 255.0
    return (x - x.mean()) / (x.std() + 1e-6)

## The model

Copy your `UNet` class (and `DoubleConv`, if you wrote one) from notebook 02 into the next cell, without changes. The weights only load into exactly the same architecture.

In [ ]:
# paste the UNet class from notebook 02 here

In [ ]:
weights = sorted(glob.glob('/kaggle/input/**/unet_best.pt', recursive=True)) or sorted(glob.glob('**/unet_best.pt', recursive=True))
assert weights, 'unet_best.pt not found: add the output of notebook 02 with + Add Input'
print('weights:', weights[0])
model = UNet().to(DEVICE)
model.load_state_dict(torch.load(weights[0], map_location=DEVICE))
model.eval()

def pad16(x):
    h, w = x.shape[-2:]
    ph, pw = (-h) % 16, (-w) % 16
    return F.pad(x, (0, pw, 0, ph), mode='reflect'), h, w

@torch.no_grad()
def predict(img):
    x = torch.from_numpy(normalise(img).transpose(2, 0, 1))[None].to(DEVICE)
    xt, h, w = pad16(x)
    logits = model(xt)[..., :h, :w]
    return torch.softmax(logits, 1)[0].cpu().numpy()

## From pixels to nuclei, and the scores

The helper functions below are tested on small synthetic masks before they were put here: a perfect prediction scores 1.0, two touching nuclei predicted as one count as a merge, and one nucleus cut in two counts as a split.

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage.segmentation import watershed, relabel_sequential


def remove_small(label, min_size):
    """Drop labelled objects smaller than min_size pixels and renumber 1..n."""
    sizes = np.bincount(label.ravel())
    small = sizes < min_size
    small[0] = False
    label = label.copy()
    label[small[label]] = 0
    return relabel_sequential(label)[0]


def to_instances(prob, thr=0.5, min_size=15):
    """Pixel probabilities (3, H, W) -> instance labels (H, W).

    Seeds are the connected regions of 'interior'. Each seed grows by watershed
    over the whole nucleus area (interior + boundary), so touching nuclei that
    the boundary class separated stay separate.
    """
    interior = prob[1] > thr
    nucleus = (prob[1] + prob[2]) > thr
    seeds = remove_small(ndi.label(interior)[0], min_size // 3 + 1)
    label = watershed(-prob[1], seeds, mask=nucleus)
    return remove_small(label, min_size)


def match(true, pred):
    """IoU between every true and every predicted nucleus, plus the overlap counts."""
    true = relabel_sequential(true)[0]
    pred = relabel_sequential(pred)[0]
    nt, npred = int(true.max()), int(pred.max())
    inter = np.bincount(true.ravel().astype(np.int64) * (npred + 1) + pred.ravel(),
                        minlength=(nt + 1) * (npred + 1)).reshape(nt + 1, npred + 1)
    area_t = inter.sum(1)[1:]
    area_p = inter.sum(0)[1:]
    inter = inter[1:, 1:]
    union = area_t[:, None] + area_p[None, :] - inter
    iou = inter / np.maximum(union, 1)
    return iou, inter, area_t, area_p


def dsb_scores(iou, thresholds=np.arange(0.5, 1.0, 0.05)):
    """Data Science Bowl 2018 score for one image and F1 at IoU 0.5.

    Above IoU 0.5 a nucleus can match at most one prediction, so counting
    rows and columns with a match is the same as a one-to-one matching.
    """
    nt, npred = iou.shape
    per_t = []
    for t in thresholds:
        hit = iou > t
        tp = int(hit.any(1).sum())
        fn = nt - tp
        fp = npred - int(hit.any(0).sum())
        per_t.append(tp / (tp + fp + fn) if tp + fp + fn else 1.0)
    hit = iou > 0.5
    tp = int(hit.any(1).sum())
    fn, fp = nt - tp, npred - int(hit.any(0).sum())
    f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0
    return float(np.mean(per_t)), f1, tp, fp, fn


def merges_and_splits(inter, area_t, area_p, share=0.5):
    """Merge: one prediction covers more than half of two or more true nuclei.
    Split: one true nucleus covers more than half of two or more predictions."""
    if inter.size == 0:
        return 0, 0
    cover_t = inter / np.maximum(area_t[:, None], 1)
    cover_p = inter / np.maximum(area_p[None, :], 1)
    merges = int(((cover_t > share).sum(0) >= 2).sum())
    splits = int(((cover_p > share).sum(1) >= 2).sum())
    return merges, splits


def entropy_map(prob):
    p = np.clip(prob, 1e-8, 1.0)
    return -(p * np.log(p)).sum(0)


def warning_test(uncertainty, score, share=0.1):
    """Flag the most uncertain images and count how many of the worst images are caught."""
    from scipy.stats import hypergeom, spearmanr
    n = len(score)
    k = max(1, int(np.ceil(share * n)))
    worst = set(np.argsort(score)[:k])
    flagged = set(np.argsort(-np.asarray(uncertainty))[:k])
    caught = len(worst & flagged)
    expected = k * k / n
    p_value = float(hypergeom.sf(caught - 1, n, k, k))
    rho, rho_p = spearmanr(uncertainty, score)
    return dict(n_images=n, k=k, caught=caught, expected_by_chance=round(expected, 2),
                p_value=p_value, spearman_rho=float(rho), spearman_p=float(rho_p),
                missed=sorted(int(i) for i in worst - flagged))

In [ ]:
rows, keep = [], {}
for r in val.itertuples():
    img, true = load(paths[r.id])
    prob = predict(img)
    pred = to_instances(prob)
    iou, inter, area_t, area_p = match(true, pred)
    ap, f1, tp, fp, fn = dsb_scores(iou)
    merges, splits = merges_and_splits(inter, area_t, area_p)
    ent = entropy_map(prob)
    band = find_boundaries(pred > 0, mode='thick') | (pred > 0)
    rows.append(dict(id=r.id, type=r.type, n_true=int(true.max()), n_pred=int(pred.max()),
                     ap=ap, f1_50=f1, tp=tp, fp=fp, fn=fn, merges=merges, splits=splits,
                     entropy=float(ent.mean()),
                     entropy_nuclei=float(ent[band].mean()) if band.any() else 0.0))
    keep[r.id] = (img, true, pred)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'per_image.csv', index=False)
print(len(df), 'validation images, mean AP', round(df.ap.mean(), 3))

## H1 and H2: scores per image type, merged and split nuclei

In [ ]:
by_type = df.groupby('type').agg(images=('id', 'size'), mean_ap=('ap', 'mean'), median_ap=('ap', 'median'),
                                 mean_f1_50=('f1_50', 'mean'), nuclei=('n_true', 'sum'),
                                 merges=('merges', 'sum'), splits=('splits', 'sum'),
                                 missed=('fn', 'sum'), extra=('fp', 'sum')).round(3)
by_type.loc['all'] = [len(df), df.ap.mean(), df.ap.median(), df.f1_50.mean(), df.n_true.sum(),
                      df.merges.sum(), df.splits.sum(), df.fn.sum(), df.fp.sum()]
by_type = by_type.round(3)
by_type.to_csv(OUT / 'by_type.csv')
print(by_type.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
types = sorted(df.type.unique())
ax.boxplot([df[df.type == t].ap for t in types])
ax.set_xticks(range(1, len(types) + 1), [f'{t}\n(n={sum(df.type == t)})' for t in types])
ax.set_ylabel('AP per image (IoU 0.5-0.95)')
ax.set_title('Score per image type')
plt.tight_layout(); plt.savefig(OUT / 'ap_by_type.png', dpi=120); plt.show()

## H3: does the uncertainty point to the bad images?

The number that was fixed in the plan is the mean pixel entropy over the whole image. The entropy over the nucleus area only is reported next to it as an extra, but the answer to H3 uses the planned number.

In [ ]:
warn = warning_test(df.entropy.values, df.ap.values)
warn_nuc = warning_test(df.entropy_nuclei.values, df.ap.values)
warn['missed_ids'] = [df.id[i] for i in warn['missed']]
summary = dict(planned_whole_image_entropy=warn,
               extra_nucleus_area_entropy={k: v for k, v in warn_nuc.items() if k != 'missed'})
(OUT / 'warning.json').write_text(json.dumps(summary, indent=2, default=float))
print(f"worst {warn['k']} images: {warn['caught']} flagged by the uncertainty, "
      f"{warn['expected_by_chance']} expected by chance (p = {warn['p_value']:.3g})")
print(f"Spearman rank correlation entropy vs AP: {warn['spearman_rho']:.2f}")
print('nucleus-area entropy (extra):', warn_nuc['caught'], 'caught,', round(warn_nuc['spearman_rho'], 2), 'rho')

fig, ax = plt.subplots(figsize=(6, 4))
for t in types:
    s = df[df.type == t]
    ax.scatter(s.entropy, s.ap, s=14, label=t)
ax.set_xlabel('mean pixel entropy (uncertainty)'); ax.set_ylabel('AP per image')
ax.legend(); ax.set_title('Uncertainty against score')
plt.tight_layout(); plt.savefig(OUT / 'uncertainty_vs_ap.png', dpi=120); plt.show()

## The worst images

Green: expert outline. Red: predicted outline. The title gives the AP and whether the uncertainty flagged the image.

In [ ]:
k = warn['k']
flagged = set(df.sort_values('entropy', ascending=False).id[:k])
worst = df.sort_values('ap').head(8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.ravel(), worst.itertuples()):
    img, true, pred = keep[r.id]
    show = img.astype(float) / max(img.max(), 1)
    show[find_boundaries(true)] = [0, 1, 0]
    show[find_boundaries(pred)] = [1, 0, 0]
    ax.imshow(show); ax.axis('off')
    ax.set_title(f"{r.type}, AP {r.ap:.2f}, {'flagged' if r.id in flagged else 'NOT flagged'}", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / 'worst_images.png', dpi=110); plt.show()

## Answers to the hypotheses

Write them here in your own words after looking at the numbers above:

- H1 (histology and brightfield lower than fluorescence):
- H2 (on fluorescence, most errors are merged or split touching nuclei):
- H3 (uncertainty catches the worst images better than chance, but misses confident mistakes):

In [ ]:
shutil.make_archive(str(WORK / 'results_03'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))